In [37]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

import warnings
warnings.filterwarnings("ignore")

print("🚀 Starting pipeline...")

# ================= LOAD =================
train_df = pd.read_pickle("train_final.pkl")
val_df   = pd.read_pickle("val_final.pkl")

print("✅ Data loaded:", train_df.shape, val_df.shape)

# ================= LABEL FIX =================
def create_label_string(row):
    aspects = row.get('aspects_parsed', [])
    sentiments = row.get('sentiments_parsed', {})

    labels = []
    for aspect in aspects:
        if aspect == "none":
            continue
        sent = sentiments.get(aspect, None)
        if sent:
            labels.append(f"{aspect}: {sent}")
    return labels

train_df['label_list'] = train_df.apply(create_label_string, axis=1)
val_df['label_list']   = val_df.apply(create_label_string, axis=1)

print("✅ Labels created")

# ================= INPUT =================
train_df['input_text'] = (
    train_df['clean_text'].fillna("") +
    " [CAT: " + train_df['business_category'].astype(str) + "]" +
    " [RATING: " + train_df['star_rating'].astype(str) + "]"
)

val_df['input_text'] = (
    val_df['clean_text'].fillna("") +
    " [CAT: " + val_df['business_category'].astype(str) + "]" +
    " [RATING: " + val_df['star_rating'].astype(str) + "]"
)

print("✅ Input text ready")

# ================= LABEL SPACE =================
all_labels = set()
for labels in train_df['label_list']:
    for l in labels:
        all_labels.add(l)

unique_labels = sorted(list(all_labels))
print("✅ Labels:", len(unique_labels))

mlb = MultiLabelBinarizer(classes=unique_labels)

train_labels = mlb.fit_transform(train_df['label_list'])
val_labels   = mlb.transform(val_df['label_list'])

print("✅ Labels binarized")

# ================= TF-IDF =================
vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 3),
    min_df=1,
    max_df=0.95
)

X_train = vectorizer.fit_transform(train_df['input_text'])
X_val   = vectorizer.transform(val_df['input_text'])

print("✅ TF-IDF ready:", X_train.shape)

# ================= MODEL =================
model = OneVsRestClassifier(
    LogisticRegression(
        max_iter=2000,
        C=3.0,
        class_weight="balanced"
    )
)

print("🚀 Training...")
model.fit(X_train, train_labels)
print("✅ Model trained")

# ================= PREDICT =================
probs = model.predict_proba(X_val)

threshold = 0.3
predictions = (probs >= threshold).astype(int)

print("✅ Predictions done")

# ================= EVAL =================
subset_acc = accuracy_score(val_labels, predictions)
f1_micro   = f1_score(val_labels, predictions, average='micro')
f1_macro   = f1_score(val_labels, predictions, average='macro')

print("\n📊 RESULTS")
print(f"Subset Accuracy: {subset_acc:.4f}")
print(f"Micro F1:        {f1_micro:.4f}")
print(f"Macro F1:        {f1_macro:.4f}")

# ================= TOP LABELS =================
precision, recall, f1, _ = precision_recall_fscore_support(
    val_labels, predictions, average=None, zero_division=0
)

results = [(unique_labels[i], f1[i]) for i in range(len(unique_labels))]
results.sort(key=lambda x: x[1], reverse=True)

print("\n🔥 Top labels:")
for r in results[:10]:
    print(f"{r[0]} → F1: {r[1]:.3f}")

# ================= SAMPLE =================
pred_labels = mlb.inverse_transform(predictions)

print("\n🧪 SAMPLE PREDICTIONS")
for i in range(5):
    print("\n---")
    print("TEXT:", val_df.iloc[i]['input_text'][:100])
    print("TRUE:", val_df.iloc[i]['label_list'])
    print("PRED:", pred_labels[i])

print("\n✅ DONE")

🚀 Starting pipeline...
✅ Data loaded: (1971, 15) (500, 15)
✅ Labels created
✅ Input text ready
✅ Labels: 24
✅ Labels binarized
✅ TF-IDF ready: (1971, 20000)
🚀 Training...
✅ Model trained
✅ Predictions done

📊 RESULTS
Subset Accuracy: 0.0760
Micro F1:        0.5350
Macro F1:        0.3153

🔥 Top labels:
service: positive → F1: 0.739
service: negative → F1: 0.642
app_experience: negative → F1: 0.593
ambiance: positive → F1: 0.570
price: negative → F1: 0.561
general: positive → F1: 0.551
delivery: negative → F1: 0.523
food: negative → F1: 0.500
app_experience: positive → F1: 0.496
food: positive → F1: 0.483

🧪 SAMPLE PREDICTIONS

---
TEXT: مريم سوتلي الاظافر تحفه اوي ❤️❤️❤️❤️❤️ [CAT: صالون تجميل] [RATING: 5]
TRUE: ['service: positive']
PRED: ('ambiance: positive', 'cleanliness: positive', 'service: positive')

---
TEXT: التطبيق جميل أتمنى إضافة البحث عن طريق الخريطة وتكون جميع العروض ظاهرة بالخريطة مثل موقع بوكينق وشكر
TRUE: ['app_experience: neutral']
PRED: ('app_experience: negative', '